In [ ]:
# Evaluación del modelo
Proyecto MIA - Resultados de detección de anomalías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#Carga de datos
df = pd.read_csv("../data/sample/ejemplo_dataset.csv", sep=";")
df.head()
#Preprocesamiento y entrenamiento
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.dropna(subset=["timestamp_utc", "value", "instance", "metric"])

df["t15"] = df["timestamp_utc"].dt.floor("15min")

interval_df = (
    df.groupby(["t15", "instance", "metric"], as_index=False)["value"]
      .mean()
)

wide = interval_df.pivot_table(
    index=["t15", "instance"],
    columns="metric",
    values="value",
    aggfunc="mean"
).reset_index()

wide = wide.sort_values(["instance", "t15"]).reset_index(drop=True)

id_cols = ["t15", "instance"]
feature_cols = [c for c in wide.columns if c not in id_cols]

for col in feature_cols:
    wide[col] = pd.to_numeric(wide[col], errors="coerce")

wide[feature_cols] = wide[feature_cols].fillna(wide[feature_cols].median(numeric_only=True))

X = wide.drop(columns=["t15", "instance"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=500,
    contamination=0.01,
    random_state=42
)
model.fit(X_scaled)

scores = -model.score_samples(X_scaled)
wide["anomaly_score"] = scores
threshold = np.percentile(scores, 97)
wide["is_anomaly"] = wide["anomaly_score"] >= threshold

wide.head()

#Anomalias
wide["is_anomaly"].value_counts()

#Mostrar registros anomalos
anomalias = wide[wide["is_anomaly"] == True]
anomalias[["t15", "instance", "anomaly_score"]]

#Grafico score de anomalia
plt.figure(figsize=(10,5))
plt.plot(wide["t15"], wide["anomaly_score"])
plt.axhline(threshold, linestyle="--")
plt.title("Puntaje de anomalía")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#Grafico por metrica
if "cpu_pct" in wide.columns:
    plt.figure(figsize=(10,5))
    plt.plot(wide["t15"], wide["cpu_pct"])
    plt.title("Comportamiento de CPU")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

if "memory_pct" in wide.columns:
    plt.figure(figsize=(10,5))
    plt.plot(wide["t15"], wide["memory_pct"])
    plt.title("Comportamiento de memoria")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Interpretación de resultados

El modelo asigna un puntaje de anomalía a cada observación. Los valores más altos representan comportamientos más alejados del patrón normal de operación.

Para identificar anomalías, se definió un umbral con base en el percentil 97 de los puntajes obtenidos. Las observaciones que superan este umbral son marcadas como anómalas.

En un escenario real, este resultado permite generar alertas tempranas sobre degradaciones en métricas de infraestructura como CPU, memoria o disco.


